# 03. KLダイバージェンスとFisher情報 - 情報幾何の核心への橋渡し

本ノートブックでは、あなたの研究経験（KLダイバージェンスによる情報利得の計算）を
情報幾何の中心概念に接続します。

## 本ノートブックの目標
- KLダイバージェンスの幾何学的意味を理解する
- Fisher情報行列がリーマン計量になる理由を把握する
- KLの2次近似がFisher計量を導くことを確認する

**これが情報幾何の最も重要な橋渡しです！**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import quad

plt.rcParams['figure.figsize'] = (10, 6)

---
## 1. KLダイバージェンスの復習

### 定義
確率分布 $p$ と $q$ のKLダイバージェンス（相対エントロピー）：
$$D_{\text{KL}}(p \| q) = \int p(x) \log \frac{p(x)}{q(x)} dx = \mathbb{E}_p\left[ \log \frac{p(x)}{q(x)} \right]$$

### 基本的性質
- **非負性**: $D_{\text{KL}}(p \| q) \geq 0$、等号は $p = q$ のときのみ
- **非対称性**: $D_{\text{KL}}(p \| q) \neq D_{\text{KL}}(q \| p)$ 一般に
- **三角不等式を満たさない**: 距離の公理を満たさない

### あなたの経験との接続
- **情報利得**: $D_{\text{KL}}(\text{posterior} \| \text{prior})$ は観測による情報量
- **ベイズ推定**: 事後分布と事前分布の「距離」の指標

In [ ]:
def kl_divergence_gaussian(mu1, sigma1, mu2, sigma2):
    """
    2つの正規分布間のKLダイバージェンス
    D_KL(N(μ1,σ1²) || N(μ2,σ2²))
    
    解析解:
    D_KL = log(σ2/σ1) + (σ1² + (μ1-μ2)²)/(2σ2²) - 1/2
    """
    return np.log(sigma2/sigma1) + (sigma1**2 + (mu1-mu2)**2)/(2*sigma2**2) - 0.5


def visualize_kl_asymmetry():
    """
    KLダイバージェンスの非対称性を可視化
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # 2つの分布
    mu_p, sigma_p = 0, 1
    mu_q, sigma_q = 2, 1.5
    
    x = np.linspace(-5, 7, 200)
    p = stats.norm.pdf(x, mu_p, sigma_p)
    q = stats.norm.pdf(x, mu_q, sigma_q)
    
    # 左: 分布の比較
    ax1 = axes[0]
    ax1.plot(x, p, 'b-', linewidth=2, label=f'p = N({mu_p}, {sigma_p}²)')
    ax1.plot(x, q, 'r-', linewidth=2, label=f'q = N({mu_q}, {sigma_q}²)')
    ax1.fill_between(x, p, alpha=0.3)
    ax1.fill_between(x, q, alpha=0.3)
    ax1.set_xlabel('x')
    ax1.set_ylabel('Density')
    ax1.set_title('Two Gaussian distributions')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 中央: D_KL(p||q)
    ax2 = axes[1]
    kl_pq = kl_divergence_gaussian(mu_p, sigma_p, mu_q, sigma_q)
    
    # 被積分関数 p(x) log(p(x)/q(x))
    log_ratio_pq = np.log(p / (q + 1e-10) + 1e-10)
    integrand_pq = p * log_ratio_pq
    
    ax2.fill_between(x, integrand_pq, where=integrand_pq > 0, color='green', alpha=0.5, label='Positive')
    ax2.fill_between(x, integrand_pq, where=integrand_pq < 0, color='red', alpha=0.5, label='Negative')
    ax2.axhline(0, color='k', linewidth=0.5)
    ax2.set_xlabel('x')
    ax2.set_ylabel('p(x) log(p/q)')
    ax2.set_title(f'D_KL(p||q) = {kl_pq:.4f}')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 右: D_KL(q||p)
    ax3 = axes[2]
    kl_qp = kl_divergence_gaussian(mu_q, sigma_q, mu_p, sigma_p)
    
    log_ratio_qp = np.log(q / (p + 1e-10) + 1e-10)
    integrand_qp = q * log_ratio_qp
    
    ax3.fill_between(x, integrand_qp, where=integrand_qp > 0, color='green', alpha=0.5, label='Positive')
    ax3.fill_between(x, integrand_qp, where=integrand_qp < 0, color='red', alpha=0.5, label='Negative')
    ax3.axhline(0, color='k', linewidth=0.5)
    ax3.set_xlabel('x')
    ax3.set_ylabel('q(x) log(q/p)')
    ax3.set_title(f'D_KL(q||p) = {kl_qp:.4f}')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"""
【KLダイバージェンスの非対称性】
D_KL(p||q) = {kl_pq:.4f}
D_KL(q||p) = {kl_qp:.4f}
差: {abs(kl_pq - kl_qp):.4f}

→ KLダイバージェンスは「距離」ではない！
→ しかし、微小変化の2次近似では対称になる（後述）
""")

visualize_kl_asymmetry()

---
## 2. KLダイバージェンスの2次近似 → Fisher情報行列

### 核心的な結果
パラメータ $\theta$ の近傍で：
$$D_{\text{KL}}(p_\theta \| p_{\theta + d\theta}) \approx \frac{1}{2} d\theta^\top I(\theta) d\theta$$

ここで $I(\theta)$ はFisher情報行列：
$$I(\theta)_{ij} = \mathbb{E}_{p_\theta}\left[ \frac{\partial \log p(x|\theta)}{\partial \theta_i} \frac{\partial \log p(x|\theta)}{\partial \theta_j} \right]$$

**これが情報幾何の出発点！**
- KLの2次近似 → **二次形式**（対称、正定値）
- この二次形式がリーマン計量を定義する

In [ ]:
def derive_fisher_from_kl():
    """
    正規分布での具体的計算：KLの2次近似 → Fisher情報行列
    """
    print("""
【導出】正規分布 N(μ, σ²) の場合

基準点: θ = (μ₀, σ₀)
近傍点: θ + dθ = (μ₀ + dμ, σ₀ + dσ)

KLダイバージェンスの解析式:
D_KL(N(μ₀,σ₀²) || N(μ₀+dμ, (σ₀+dσ)²))
  = log((σ₀+dσ)/σ₀) + (σ₀² + dμ²)/(2(σ₀+dσ)²) - 1/2

Taylor展開（2次まで）:
  ≈ dσ/σ₀ + (σ₀² + dμ²)/(2σ₀²)(1 - 2dσ/σ₀) - 1/2
  ≈ dσ/σ₀ + 1/2 + dμ²/(2σ₀²) - dσ/σ₀ - 1/2
  ≈ dμ²/(2σ₀²) + dσ²/σ₀² + O(dθ³)

行列形式で:
D_KL ≈ (1/2) [dμ, dσ] [[1/σ₀², 0], [0, 2/σ₀²]] [dμ, dσ]ᵀ

したがって Fisher情報行列は:
I(μ, σ) = [[1/σ², 0   ],
           [0,    2/σ²]]
""")

derive_fisher_from_kl()

In [ ]:
def visualize_kl_quadratic_approximation():
    """
    KLダイバージェンスと2次近似（Fisher計量）の比較
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 基準点
    mu0, sigma0 = 0, 1
    
    # μ方向の断面
    ax1 = axes[0]
    dmu_range = np.linspace(-2, 2, 100)
    
    # 真のKL
    kl_true = [kl_divergence_gaussian(mu0, sigma0, mu0 + dmu, sigma0) for dmu in dmu_range]
    
    # 2次近似: (1/2) dμ² / σ₀²
    kl_approx = 0.5 * dmu_range**2 / sigma0**2
    
    ax1.plot(dmu_range, kl_true, 'b-', linewidth=2, label='True KL')
    ax1.plot(dmu_range, kl_approx, 'r--', linewidth=2, label='Quadratic approx (Fisher)')
    ax1.set_xlabel('dμ')
    ax1.set_ylabel('D_KL')
    ax1.set_title(f'KL divergence along μ direction (σ₀={sigma0})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2D等高線プロット
    ax2 = axes[1]
    dmu_grid = np.linspace(-1, 1, 50)
    dsigma_grid = np.linspace(-0.5, 0.5, 50)
    DMU, DSIGMA = np.meshgrid(dmu_grid, dsigma_grid)
    
    # σ + dσ > 0 を確保
    valid = (sigma0 + DSIGMA) > 0.1
    
    KL_TRUE = np.zeros_like(DMU)
    KL_APPROX = np.zeros_like(DMU)
    
    for i in range(len(dmu_grid)):
        for j in range(len(dsigma_grid)):
            if valid[j, i]:
                KL_TRUE[j, i] = kl_divergence_gaussian(mu0, sigma0, 
                                                       mu0 + dmu_grid[i], 
                                                       sigma0 + dsigma_grid[j])
                # 2次近似: (1/2)(dμ²/σ² + 2dσ²/σ²)
                KL_APPROX[j, i] = 0.5 * (dmu_grid[i]**2 / sigma0**2 + 
                                         2 * dsigma_grid[j]**2 / sigma0**2)
    
    # 真のKL等高線（青）
    levels = [0.05, 0.1, 0.2, 0.5]
    cs1 = ax2.contour(DMU, DSIGMA, KL_TRUE, levels=levels, colors='blue', linestyles='-')
    ax2.clabel(cs1, inline=True, fontsize=8, fmt='%.2f')
    
    # 2次近似等高線（赤破線）
    cs2 = ax2.contour(DMU, DSIGMA, KL_APPROX, levels=levels, colors='red', linestyles='--')
    
    ax2.plot(0, 0, 'ko', markersize=10)
    ax2.set_xlabel('dμ')
    ax2.set_ylabel('dσ')
    ax2.set_title('KL contours: True (blue) vs Fisher approx (red dashed)')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("""
【観察ポイント】
1. 原点付近では青線と赤破線がほぼ一致 → 2次近似が良い
2. 遠くでは乖離 → 高次項の影響
3. Fisher計量はKLの「局所的」な振る舞いを捉える
4. この楕円形がリーマン計量の「単位球」
""")

visualize_kl_quadratic_approximation()

---
## 3. Fisher情報行列の性質

### 3つの等価な定義

1. **スコア関数の共分散**:
$$I(\theta)_{ij} = \mathbb{E}\left[ \frac{\partial \log p}{\partial \theta_i} \frac{\partial \log p}{\partial \theta_j} \right]$$

2. **対数尤度のヘッセ行列の期待値（の負）**:
$$I(\theta)_{ij} = -\mathbb{E}\left[ \frac{\partial^2 \log p}{\partial \theta_i \partial \theta_j} \right]$$

3. **KLダイバージェンスの2次係数**（前節）

### 重要な性質
- **正定値**（正則モデルで）→ リーマン計量になれる
- **座標変換で共変**（テンソル的に変換）
- **Cramér-Raoの下界**: $\text{Var}(\hat{\theta}) \geq I(\theta)^{-1}$

In [ ]:
def compute_fisher_three_ways():
    """
    正規分布のFisher情報行列を3つの方法で計算
    """
    print("="*60)
    print("正規分布 N(μ, σ²) のFisher情報行列")
    print("="*60)
    
    print("""
【方法1: スコア関数の共分散】

対数尤度: log p(x|μ,σ) = -log(σ√2π) - (x-μ)²/(2σ²)

スコア関数:
  ∂log p/∂μ = (x-μ)/σ²
  ∂log p/∂σ = -1/σ + (x-μ)²/σ³

期待値:
  E[(∂log p/∂μ)²] = E[(x-μ)²]/σ⁴ = σ²/σ⁴ = 1/σ²
  E[(∂log p/∂σ)²] = E[(-1/σ + (x-μ)²/σ³)²]
                  = 1/σ² - 2E[(x-μ)²]/σ⁴ + E[(x-μ)⁴]/σ⁶
                  = 1/σ² - 2/σ² + 3σ⁴/σ⁶  (E[(x-μ)⁴] = 3σ⁴)
                  = 2/σ²
  E[(∂log p/∂μ)(∂log p/∂σ)] = E[(x-μ)/σ² · (-1/σ + (x-μ)²/σ³)]
                            = -E[(x-μ)]/σ³ + E[(x-μ)³]/σ⁵
                            = 0  (奇数次モーメントは0)

→ I = [[1/σ², 0], [0, 2/σ²]]
""")
    
    print("""
【方法2: ヘッセ行列の負の期待値】

2階微分:
  ∂²log p/∂μ² = -1/σ²
  ∂²log p/∂σ² = 1/σ² - 3(x-μ)²/σ⁴
  ∂²log p/∂μ∂σ = -2(x-μ)/σ³

期待値の負:
  -E[∂²log p/∂μ²] = 1/σ²
  -E[∂²log p/∂σ²] = -1/σ² + 3σ²/σ⁴ = 2/σ²
  -E[∂²log p/∂μ∂σ] = 0

→ I = [[1/σ², 0], [0, 2/σ²]]  （方法1と一致）
""")
    
    print("""
【方法3: KLダイバージェンスの2次係数】

前節で示した通り:
D_KL(N(μ,σ²) || N(μ+dμ, (σ+dσ)²)) ≈ (1/2)[dμ, dσ] I [dμ, dσ]ᵀ

→ I = [[1/σ², 0], [0, 2/σ²]]  （一致）
""")

compute_fisher_three_ways()

In [ ]:
def visualize_fisher_metric_ellipses():
    """
    パラメータ空間の各点でFisher計量楕円を可視化
    σが小さい領域ほど楕円が小さい（距離が大きい）
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # パラメータ空間の点
    mu_points = np.linspace(-2, 2, 5)
    sigma_points = np.linspace(0.5, 2, 4)
    
    theta = np.linspace(0, 2*np.pi, 100)
    scale = 0.15  # 楕円のスケール
    
    for mu in mu_points:
        for sigma in sigma_points:
            # Fisher計量: I = diag(1/σ², 2/σ²)
            # 単位楕円: dμ²/σ² + 2dσ²/σ² = 1
            # dμ = σ cos(θ), dσ = σ/√2 sin(θ)
            
            ellipse_dmu = sigma * np.cos(theta) * scale
            ellipse_dsigma = sigma / np.sqrt(2) * np.sin(theta) * scale
            
            ax.plot(mu + ellipse_dmu, sigma + ellipse_dsigma, 'b-', linewidth=1)
            ax.plot(mu, sigma, 'ko', markersize=4)
    
    ax.set_xlabel('μ', fontsize=12)
    ax.set_ylabel('σ', fontsize=12)
    ax.set_title('Fisher metric ellipses in Gaussian parameter space\n(Smaller ellipse = larger "distance" per coordinate change)', fontsize=11)
    ax.set_xlim(-3, 3)
    ax.set_ylim(0, 2.5)
    ax.grid(True, alpha=0.3)
    
    # 注釈
    ax.annotate('Small σ: ellipses are small\n→ high information density',
                xy=(0, 0.5), xytext=(1.5, 0.3),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=10, color='red')
    
    plt.tight_layout()
    plt.show()
    
    print("""
【Fisher計量楕円の解釈】
- 楕円が小さい = 同じ座標変化でも「情報的距離」が大きい
- σが小さい領域：分布が鋭いので、微小な変化でも大きな情報変化
- これがカルマンフィルタで観測精度が高いほど更新が効く理由の幾何学的説明
""")

visualize_fisher_metric_ellipses()

---
## 4. 情報利得とKLダイバージェンス

### あなたの研究経験との接続

ベイズ更新での**情報利得**:
$$\text{Info Gain} = D_{\text{KL}}(p(\theta|x) \| p(\theta))$$

これは「観測 $x$ によって得られた情報量」を測る。

### 幾何学的解釈
- 事前分布から事後分布への「移動距離」（の非対称版）
- Fisher計量で測ると、より「真の」距離に近づく

In [ ]:
def visualize_information_gain():
    """
    ベイズ更新における情報利得の可視化
    あなたの研究経験（情報利得の計算）との接続
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 設定：μの推定（σは既知）
    sigma_known = 1.0
    
    # 事前分布: N(μ_prior, τ_prior²)
    mu_prior, tau_prior = 0, 2
    
    # 異なる観測値での情報利得を比較
    observations = [0.5, 1.0, 2.0, 3.0]
    
    mu_range = np.linspace(-4, 6, 200)
    
    ax1 = axes[0]
    prior = stats.norm.pdf(mu_range, mu_prior, tau_prior)
    ax1.plot(mu_range, prior, 'k--', linewidth=2, label='Prior')
    
    info_gains = []
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(observations)))
    
    for x_obs, color in zip(observations, colors):
        # 事後分布の計算（正規-正規共役）
        precision_prior = 1 / tau_prior**2
        precision_likelihood = 1 / sigma_known**2
        precision_post = precision_prior + precision_likelihood
        mu_post = (precision_prior * mu_prior + precision_likelihood * x_obs) / precision_post
        tau_post = 1 / np.sqrt(precision_post)
        
        posterior = stats.norm.pdf(mu_range, mu_post, tau_post)
        ax1.plot(mu_range, posterior, color=color, linewidth=1.5, 
                 label=f'Posterior (x={x_obs})')
        
        # 情報利得: D_KL(posterior || prior)
        info_gain = kl_divergence_gaussian(mu_post, tau_post, mu_prior, tau_prior)
        info_gains.append(info_gain)
    
    ax1.set_xlabel('μ')
    ax1.set_ylabel('p(μ)')
    ax1.set_title('Bayesian update with different observations')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 情報利得のプロット
    ax2 = axes[1]
    ax2.bar(observations, info_gains, color=colors, width=0.3)
    ax2.set_xlabel('Observation x')
    ax2.set_ylabel('Information Gain (nats)')
    ax2.set_title('D_KL(posterior || prior)')
    ax2.grid(True, alpha=0.3)
    
    for x, ig in zip(observations, info_gains):
        ax2.text(x, ig + 0.05, f'{ig:.3f}', ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print("""
【情報利得の解釈】
- 事前分布から遠い観測ほど情報利得が大きい
- これは直感と一致：予想外の観測は多くの情報をもたらす
- Fisher計量の視点：パラメータ空間での「移動距離」
""")

visualize_information_gain()

---
## 5. 自然勾配への橋渡し（予告）

### 問題：通常の勾配降下は座標依存
$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

同じ点を (μ, σ) で表しても (μ, σ²) で表しても、勾配の大きさが変わる。

### 解決：自然勾配
$$\theta_{t+1} = \theta_t - \eta I(\theta_t)^{-1} \nabla L(\theta_t)$$

Fisher計量の逆行列をかけることで、座標に依存しない「真の最急降下方向」を得る。

In [ ]:
def compare_gradient_vs_natural_gradient():
    """
    通常勾配と自然勾配の違いを示す
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 設定：損失関数（例：真の分布からのKL）
    mu_true, sigma_true = 2, 1
    
    # 勾配降下のシミュレーション
    def loss(mu, sigma):
        """損失 = D_KL(true || current)"""
        return kl_divergence_gaussian(mu_true, sigma_true, mu, sigma)
    
    def gradient(mu, sigma):
        """損失の勾配（数値微分）"""
        eps = 1e-5
        grad_mu = (loss(mu + eps, sigma) - loss(mu - eps, sigma)) / (2 * eps)
        grad_sigma = (loss(mu, sigma + eps) - loss(mu, sigma - eps)) / (2 * eps)
        return np.array([grad_mu, grad_sigma])
    
    def fisher_matrix(mu, sigma):
        """Fisher情報行列"""
        return np.array([[1/sigma**2, 0], [0, 2/sigma**2]])
    
    # 初期点
    mu0, sigma0 = -1, 2
    
    # 通常勾配降下
    eta = 0.3
    mu_gd, sigma_gd = mu0, sigma0
    trajectory_gd = [(mu_gd, sigma_gd)]
    
    for _ in range(20):
        grad = gradient(mu_gd, sigma_gd)
        mu_gd -= eta * grad[0]
        sigma_gd -= eta * grad[1]
        sigma_gd = max(sigma_gd, 0.1)  # σ > 0 を保証
        trajectory_gd.append((mu_gd, sigma_gd))
    
    # 自然勾配降下
    eta_natural = 0.3
    mu_ng, sigma_ng = mu0, sigma0
    trajectory_ng = [(mu_ng, sigma_ng)]
    
    for _ in range(20):
        grad = gradient(mu_ng, sigma_ng)
        I_inv = np.linalg.inv(fisher_matrix(mu_ng, sigma_ng))
        natural_grad = I_inv @ grad
        mu_ng -= eta_natural * natural_grad[0]
        sigma_ng -= eta_natural * natural_grad[1]
        sigma_ng = max(sigma_ng, 0.1)
        trajectory_ng.append((mu_ng, sigma_ng))
    
    trajectory_gd = np.array(trajectory_gd)
    trajectory_ng = np.array(trajectory_ng)
    
    # 左図：軌跡の比較
    ax1 = axes[0]
    ax1.plot(trajectory_gd[:, 0], trajectory_gd[:, 1], 'b.-', 
             markersize=8, label='Gradient descent')
    ax1.plot(trajectory_ng[:, 0], trajectory_ng[:, 1], 'r.-', 
             markersize=8, label='Natural gradient')
    ax1.plot(mu0, sigma0, 'ks', markersize=12, label='Start')
    ax1.plot(mu_true, sigma_true, 'g*', markersize=15, label='Target')
    
    ax1.set_xlabel('μ')
    ax1.set_ylabel('σ')
    ax1.set_title('Optimization trajectories')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-2, 3)
    ax1.set_ylim(0, 3)
    
    # 右図：損失の推移
    ax2 = axes[1]
    losses_gd = [loss(mu, sigma) for mu, sigma in trajectory_gd]
    losses_ng = [loss(mu, sigma) for mu, sigma in trajectory_ng]
    
    ax2.plot(losses_gd, 'b.-', label='Gradient descent')
    ax2.plot(losses_ng, 'r.-', label='Natural gradient')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Loss (KL divergence)')
    ax2.set_title('Loss convergence')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("""
【自然勾配の利点】
1. パラメータ空間の「曲がり」を考慮した更新
2. 座標の取り方に依存しない
3. 収束が速いことが多い
4. ニューラルネットの学習（K-FAC等）で実際に使用されている
""")

compare_gradient_vs_natural_gradient()

---
## 6. 確認問題

### Q1. KLダイバージェンス
$D_{\text{KL}}(N(0,1) \| N(1,1))$ と $D_{\text{KL}}(N(1,1) \| N(0,1))$ を計算し、非対称性を確認せよ。

### Q2. Fisher情報行列
ベルヌーイ分布 $\text{Ber}(p)$ のFisher情報を計算せよ。

### Q3. 自然勾配
正規分布のパラメータ (μ, σ) で、通常勾配が (1, 1) のとき、σ=2 での自然勾配を計算せよ。

In [ ]:
# Q1の解答
kl_01 = kl_divergence_gaussian(0, 1, 1, 1)
kl_10 = kl_divergence_gaussian(1, 1, 0, 1)
print(f"Q1: D_KL(N(0,1)||N(1,1)) = {kl_01:.4f}")
print(f"    D_KL(N(1,1)||N(0,1)) = {kl_10:.4f}")
print(f"    Difference: {abs(kl_01 - kl_10):.4f}")
print("    (σが同じなので対称。μの差のみが効いている)")

In [ ]:
# Q2の解答
print("""
Q2: ベルヌーイ分布 Ber(p) のFisher情報

log p(x|p) = x log p + (1-x) log(1-p)

スコア関数:
d log p / dp = x/p - (1-x)/(1-p)

Fisher情報:
I(p) = E[(d log p / dp)²]
     = E[x²/p² - 2x(1-x)/(p(1-p)) + (1-x)²/(1-p)²]
     = p/p² + (1-p)/(1-p)²  (E[x]=p, E[x²]=p を使用)
     = 1/p + 1/(1-p)
     = 1/(p(1-p))

→ I(p) = 1/(p(1-p))

p=0.5 で最小（最も推定が難しい）、p→0 または p→1 で最大
""")

In [ ]:
# Q3の解答
sigma = 2
grad = np.array([1, 1])
I = np.array([[1/sigma**2, 0], [0, 2/sigma**2]])
I_inv = np.linalg.inv(I)
natural_grad = I_inv @ grad

print(f"Q3: σ={sigma} での自然勾配")
print(f"    通常勾配: {grad}")
print(f"    Fisher情報行列 I = {I}")
print(f"    I⁻¹ = {I_inv}")
print(f"    自然勾配 = I⁻¹ @ grad = {natural_grad}")
print(f"\n    → μ方向は σ² = 4 倍に拡大、σ方向は σ²/2 = 2 倍に拡大")

---
## まとめ

| 概念 | 定義 | 情報幾何での役割 |
|-----|------|----------------|
| KLダイバージェンス | $D_{\text{KL}}(p\|q) = E_p[\log(p/q)]$ | ダイバージェンス関数 |
| Fisher情報行列 | $I(\theta) = E[(\nabla \log p)^2]$ | リーマン計量 |
| KLの2次近似 | $D_{\text{KL}} \approx \frac{1}{2} d\theta^T I d\theta$ | 計量との関係 |
| 自然勾配 | $\tilde{\nabla} = I^{-1} \nabla$ | 座標不変な更新 |

### これで第1章への準備が整いました！

次章からは、これらの概念を微分幾何の言葉で厳密に定式化していきます。

---
**次のノートブック**: `../ch01_differential_geometry/sec01_manifold_basics.ipynb` - 多様体の基礎